# NUMPY MASTERY — CHAPTERS 18–19

```
NUMPY MASTERY
Current Chapters: 18 (Concatenation and Array Combining), 19 (Splitting Arrays)
Progress:         19 / 75 chapters
Completed:        Ch1-17
Remaining:        56 chapters
```

Combining datasets (merging batches, stacking feature columns) and splitting them (train/test,
mini-batches, cross-validation folds) are two sides of the same shape-manipulation coin.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 18 — Concatenation and Array Combining

## Concept
`np.concatenate()` joins arrays along an **existing** axis (no new axis is created — the arrays
must already agree on all other dimensions). `np.stack()` joins arrays along a **new** axis
(all inputs must have identical shape; the result has one more dimension than the inputs).

This is the single most important distinction in this chapter — get it, and `vstack`/`hstack`/
`dstack`/`column_stack` all become obvious special cases rather than things to memorize.

## Why it matters
Merging batches of data loaded separately, combining feature columns computed by different
steps of a pipeline, stacking individual samples into a batch tensor for a model.


In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

concatenated = np.concatenate([a, b])       # joins along existing axis 0 -> stays 1D
stacked = np.stack([a, b])                  # creates a NEW axis 0 -> becomes 2D

print("a.shape, b.shape           :", a.shape, b.shape)
print("np.concatenate([a, b])     :", concatenated, " shape:", concatenated.shape)
print("np.stack([a, b])           :\n", stacked, " shape:", stacked.shape)
print("np.stack([a, b], axis=1)   :\n", np.stack([a, b], axis=1), " shape:", np.stack([a, b], axis=1).shape)


### Expected Output
```
a.shape, b.shape           : (3,) (3,)
np.concatenate([a, b])     : [1 2 3 4 5 6]   shape: (6,)
np.stack([a, b])           :
 [[1 2 3]
  [4 5 6]]   shape: (2, 3)
np.stack([a, b], axis=1)   :
 [[1 4]
  [2 5]
  [3 6]]   shape: (3, 2)
```

### The core distinction, precisely
- `concatenate([a, b], axis=k)`: **total dimension count stays the same**; the sizes along
  axis `k` add together; all *other* dimensions must already match exactly.
- `stack([a, b], axis=k)`: **total dimension count increases by 1**; a brand new axis is
  inserted at position `k`; all inputs must have the **exact same shape** (concatenate has no
  such requirement beyond axis `k`).

```
concatenate: (3,) + (3,) --[axis=0]--> (6,)      <- same ndim, axis 0 grows
stack:       (3,) + (3,) --[axis=0]--> (2, 3)     <- ndim increases by 1, new axis inserted
```


In [ ]:
# --- 2D examples ---
m1 = np.array([[1, 2], [3, 4]])   # shape (2, 2)
m2 = np.array([[5, 6], [7, 8]])   # shape (2, 2)

print("m1:\n", m1)
print("m2:\n", m2)
print()
print("np.concatenate([m1, m2], axis=0):\n", np.concatenate([m1, m2], axis=0),
      " shape:", np.concatenate([m1, m2], axis=0).shape, " <- stacks ROWS (grows axis 0)")
print()
print("np.concatenate([m1, m2], axis=1):\n", np.concatenate([m1, m2], axis=1),
      " shape:", np.concatenate([m1, m2], axis=1).shape, " <- stacks COLUMNS (grows axis 1)")
print()
print("np.vstack([m1, m2]) == concatenate(axis=0)?", np.array_equal(np.vstack([m1, m2]), np.concatenate([m1, m2], axis=0)))
print("np.hstack([m1, m2]) == concatenate(axis=1)?", np.array_equal(np.hstack([m1, m2]), np.concatenate([m1, m2], axis=1)))


### Expected Output
```
m1:
 [[1 2]
  [3 4]]
m2:
 [[5 6]
  [7 8]]

np.concatenate([m1, m2], axis=0):
 [[1 2]
  [3 4]
  [5 6]
  [7 8]]   shape: (4, 2)  <- stacks ROWS (grows axis 0)

np.concatenate([m1, m2], axis=1):
 [[1 2 5 6]
  [3 4 7 8]]   shape: (2, 4)  <- stacks COLUMNS (grows axis 1)

np.vstack([m1, m2]) == concatenate(axis=0)? True
np.hstack([m1, m2]) == concatenate(axis=1)? True
```

### Every "stack" convenience function is just `stack`/`concatenate` with a fixed axis
| Function | Equivalent to |
|---|---|
| `np.vstack([a, b])` | `np.concatenate([a, b], axis=0)` (for 2D+); for 1D inputs it behaves like `stack` along a new axis 0 |
| `np.hstack([a, b])` | `np.concatenate([a, b], axis=1)` for 2D+; `concatenate(axis=0)` for 1D |
| `np.dstack([a, b])` | stack along a new/3rd axis (depth) — commonly used to combine grayscale channels into RGB |
| `np.column_stack([a, b])` | turns 1D arrays into columns of a 2D result — handy for combining separate 1D feature arrays into a feature matrix |

**Rule of thumb:** when unsure, reach for `np.concatenate(..., axis=k)` explicitly — it's the
one true primitive, and the named convenience functions are just for readability in the common
2D cases.

### Common Mistake
```python
np.concatenate([m1, m2], axis=2)   # ValueError -- m1, m2 only have 2 axes (0 and 1)!
np.stack([np.array([1,2,3]), np.array([1,2])])   # ValueError -- shapes must match EXACTLY for stack
```
`concatenate` requires all non-joining axes to match exactly; `stack` requires ALL axes/shapes
to match exactly (since it's creating a uniform new axis). Mismatched shapes fail loudly here —
good — but the error can be confusing if you don't already know this shape-matching rule.

### Interview Insight
> "What's the real difference between `concatenate` and `stack`?"
`concatenate` joins along an *existing* axis, keeping the number of dimensions the same;
`stack` creates a *new* axis, increasing the number of dimensions by one, and requires all
inputs to share the exact same shape. `vstack`/`hstack`/`dstack` are just `concatenate`/`stack`
with axis pre-selected for common 2D/3D cases.


---
# CHAPTER 19 — Splitting Arrays

## Concept
`np.split()` divides an array into **equal-sized** sub-arrays along an axis (raises an error if
it doesn't divide evenly); `np.array_split()` allows uneven splits (some resulting chunks get
one extra element). `hsplit`/`vsplit`/`dsplit` are axis-fixed convenience wrappers, mirroring
the stack functions from Chapter 18.

## Why it matters
Mini-batch generation for training, cross-validation fold creation, and separating a
feature-matrix-plus-target array (loaded together) back into `X` and `y`.


In [ ]:
data = np.arange(12)

even_split = np.split(data, 3)                 # 12 / 3 = 4 -- divides evenly
print("np.split(data, 3):", even_split)

try:
    np.split(data, 5)                          # 12 / 5 is NOT an integer -- raises!
except ValueError as e:
    print("\nnp.split(data, 5) raised ValueError:", e)

uneven_split = np.array_split(data, 5)          # tolerates uneven division
print("\nnp.array_split(data, 5):", uneven_split, "  <- earlier chunks absorb the extra elements")


### Expected Output
```
np.split(data, 3): [array([0, 1, 2, 3]), array([4, 5, 6, 7]), array([ 8,  9, 10, 11])]

np.split(data, 5) raised ValueError: array split does not result in an equal division

np.array_split(data, 5): [array([0, 1, 2]), array([3, 4, 5]), array([6, 7]), array([8, 9]), array([10, 11])]  <- earlier chunks absorb the extra elements
```

### `split` vs `array_split`
`split` is strict (equal chunks required, error otherwise) — use it when uneven chunks would
indicate a bug (e.g., you expect exactly-sized mini-batches and dropped the remainder
elsewhere). `array_split` is lenient — use it when uneven chunks are fine (e.g., the last
mini-batch of an epoch is naturally smaller than the rest). When the split isn't perfectly
even, `array_split` distributes the extra elements across the **earliest** chunks first (so
chunk sizes differ by at most 1, front-loaded), rather than dumping everything extra into the
last chunk.


In [ ]:
# --- Splitting a combined dataset back into X and y ---
combined = np.arange(1, 21).reshape(4, 5)   # imagine: 4 samples, first 4 cols = features, last col = label
print("combined:\n", combined)

X, y = np.hsplit(combined, [4])   # split at column index 4 -> first 4 cols, remaining cols
print("\nX (features):\n", X, " shape:", X.shape)
print("y (labels)  :\n", y, " shape:", y.shape)

# --- Mini-batch generation, tolerating an uneven final batch ---
dataset = np.arange(23)   # 23 samples -- doesn't divide evenly into batches of 5
batch_size = 5
n_batches = int(np.ceil(len(dataset) / batch_size))
batches = np.array_split(dataset, n_batches)
print("\nmini-batches (batch_size=5, 23 samples):")
for i, b in enumerate(batches):
    print(f"  batch {i}: {b}  (size {len(b)})")


### Expected Output
```
combined:
 [[ 1  2  3  4  5]
  [ 6  7  8  9 10]
  [11 12 13 14 15]
  [16 17 18 19 20]]

X (features):
 [[ 1  2  3  4]
  [ 6  7  8  9]
  [11 12 13 14]
  [16 17 18 19]]   shape: (4, 4)
y (labels)  :
 [[ 5]
  [10]
  [15]
  [20]]   shape: (4, 1)

mini-batches (batch_size=5, 23 samples):
  batch 0: [0 1 2 3 4]  (size 5)
  batch 1: [5 6 7 8 9]  (size 5)
  batch 2: [10 11 12 13 14]  (size 5)
  batch 3: [15 16 17 18]  (size 4)
  batch 4: [19 20 21 22]  (size 4)
```

Note `array_split` spreads any remainder across the earliest batches (sizes `[5,5,5,4,4]` here,
not `[5,5,5,5,3]`) — the batches are as equal as possible, not "last one absorbs everything."

### `np.hsplit(arr, [4])` — splitting at explicit index positions
Passing a **list of integers** (instead of a single count) to `split`/`array_split`/`hsplit`/
etc. specifies explicit split *points* rather than an equal count — `[4]` means "split once,
right after index 4," giving you a natural `X`/`y` separation for combined datasets loaded as
one array (e.g., from a CSV with the label in the last column).

### Real-world use: K-Fold cross-validation, built from scratch
```python
indices = np.random.default_rng(0).permutation(len(dataset))
folds = np.array_split(indices, k)   # k roughly-equal folds of shuffled indices
for i in range(k):
    val_idx = folds[i]
    train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
```
This is literally how `sklearn.model_selection.KFold` works conceptually — `array_split` handles
the "doesn't divide evenly" case gracefully, exactly like the mini-batch example above.

### Common Mistake
Using `np.split()` (the strict version) inside batch-generation code where the dataset size
isn't guaranteed to be a multiple of the batch size — this crashes on the very last batch of
almost every real dataset. Use `array_split` (or handle the remainder explicitly) for
production data loaders.

### Interview Insight
> "How would you generate mini-batches from a dataset of arbitrary size N given a fixed batch
> size?"
`np.array_split(data, int(np.ceil(N / batch_size)))` — tolerates a smaller final batch instead
of crashing (as strict `np.split` would) when `N` isn't a multiple of `batch_size`.


---
## Chapters 18–19 Summary

- **Concatenation vs stacking**: `concatenate` joins along an *existing* axis (same ndim in,
  same ndim out); `stack` creates a *new* axis (ndim increases by 1, requires identical input
  shapes). `vstack`/`hstack`/`dstack`/`column_stack` are just axis-fixed convenience wrappers
  around these two primitives.
- **Splitting**: `split` is strict (requires equal division, errors otherwise); `array_split`
  tolerates uneven division (some chunks get one extra element) — the correct default for
  mini-batch generation and K-Fold cross-validation on arbitrary dataset sizes.
- Passing a **list** of split points (rather than a count) to any split function gives you
  explicit control — e.g., separating a combined `X`/`y` array at a specific column index.

---

## Mastery Exercises

**Exercise 1 (Prediction):** For `a.shape == (5, 3)` and `b.shape == (5, 3)`, predict the
resulting shape of: `np.concatenate([a, b], axis=0)`, `np.concatenate([a, b], axis=1)`,
`np.stack([a, b], axis=0)`, `np.stack([a, b], axis=2)`.

**Exercise 2:** You loaded two separately-collected batches of sensor images, each of shape
`(16, 64, 64, 3)` (batch, height, width, channels). Write the call that combines them into a
single `(32, 64, 64, 3)` batch — is this a `concatenate` or a `stack` situation, and why?

**Coding Challenge:** Implement `k_fold_indices(n_samples, k, seed)` returning a list of `k`
`(train_idx, val_idx)` index-array pairs using shuffling (`np.random.default_rng(seed
).permutation`) and `np.array_split`, correctly handling `n_samples` not divisible by `k`.

**Debugging Challenge:**
```python
batch1 = np.random.default_rng(0).normal(size=(10, 5))
batch2 = np.random.default_rng(1).normal(size=(12, 5))
combined = np.stack([batch1, batch2])   # a teammate expected shape (22, 5) but got an error
```
Explain exactly why this raises an error, and give the one-line fix.

**Interview Questions:**
1. Precisely state the difference between `concatenate` and `stack` in terms of dimensionality.
2. When would `np.split()` raising an error actually be the *desired* behavior versus a bug you
   need to work around with `array_split()`?
3. How do you split a combined feature+label array into `X` and `y` using explicit split
   points?
4. Why might mini-batch generation code that uses strict `np.split()` crash unpredictably on
   some datasets but not others?

**Quiz:**
- Q1: True/False — `np.stack` requires all input arrays to have the exact same shape.
- Q2: `np.vstack` on two `(3,)` 1D arrays returns what shape?
- Q3: What does passing `[4]` (a one-element list) instead of `4` (an integer) to `np.split`
  actually mean?

---
